In [1]:
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report
import time

In [4]:
# Read the dataset
df = pd.read_csv(r'D:\Downloads\ml_features_and_labels.csv')

# Use the provided split column to create train/test sets and exclude metadata
# columns that would leak information about the label.
feature_cols = [c for c in df.columns if c not in ['label', 'ID', 'split', 'taxonomy']]

X_train = df.loc[df['split'] == 'train', feature_cols]
y_train = df.loc[df['split'] == 'train', 'label']

X_test = df.loc[df['split'] == 'test', feature_cols]
y_test = df.loc[df['split'] == 'test', 'label']

In [139]:
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight=(neg_count / pos_count)
scale_pos_weight

6.986027944111776

In [207]:
# Initialize the model
model = xgb.XGBClassifier(
    n_estimators=800,
    max_depth=9,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    enable_categorical=True,
)
start_time = time.perf_counter()

# Train the model
model.fit(X_train, y_train)
end_time = time.perf_counter()
print(f"Training time: {end_time - start_time:.10f} seconds")

Training time: 0.8609036000 seconds


In [208]:
start_time = time.perf_counter()

# Predict on the held-out test split
y_pred = model.predict(X_test)
end_time = time.perf_counter()
print(f"Prediction time: {end_time - start_time:.10f} seconds")

Prediction time: 0.0175643000 seconds


In [209]:
print('Accuracy:', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9651337165708573
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      7000
           1       0.88      0.84      0.86      1002

    accuracy                           0.97      8002
   macro avg       0.93      0.91      0.92      8002
weighted avg       0.96      0.97      0.96      8002

